In [4]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import sys
import os

try:
    current_file_path = os.path.abspath(__file__)
    current_script_dir = os.path.dirname(current_file_path)
except NameError:
    current_script_dir = os.getcwd()

# Walk up to GlobalLocal/
project_root = os.path.abspath(os.path.join(current_script_dir, '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from types import SimpleNamespace
from functools import partial
from scipy.stats import ttest_ind, ttest_rel

from ieeg.io import get_data
from ieeg.calc.fast import mean_diff
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from joblib import Parallel, delayed

from src.analysis.config import experiment_conditions
from src.analysis.config.plotting_parameters import plotting_parameters as plot_params
from src.analysis.config.condition_registry import (
    get_comparisons, get_conditions_obj, get_subtraction_pairs,
    get_anova_factors, get_anova_interactions,
)
import src.analysis.utils.general_utils as utils
from src.analysis.power.power_traces import (
    make_multi_channel_evokeds_for_all_conditions_and_rois,
    process_windowed_data_for_anova,
    create_windowed_anova_dataframe,
    _fit_anova_one_window,
    _get_factor_levels,
    compute_subcell_evoked_data,
    plot_2way_interaction_for_roi,
    plot_16_conditions_with_interaction_clusters_for_roi,
)
from src.analysis.decoding.decoding import (
    find_significant_clusters_of_series_vs_distribution_based_on_percentile,
)

print("project_root:", project_root)

['/hpc/group/coganlab/jz421/GlobalLocal/src', '/hpc/group/coganlab/jz421/GlobalLocal', '/hpc/home/jz421/miniconda3/envs/ieeg/lib/python311.zip', '/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11', '/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/lib-dynload', '', '/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/site-packages']


/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)



['/hpc/group/coganlab/jz421/GlobalLocal/src', '/hpc/group/coganlab/jz421/GlobalLocal', '/hpc/home/jz421/miniconda3/envs/ieeg/lib/python311.zip', '/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11', '/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/lib-dynload', '', '/hpc/home/jz421/miniconda3/envs/ieeg/lib/python3.11/site-packages', 'C:/Users/jz421/Desktop/GlobalLocal/IEEG_Pipelines/']
project_root: /hpc/group/coganlab/jz421/GlobalLocal


In [28]:
LAB_root = None  # auto-detect below
TASK = 'GlobalLocal'

SUBJECTS = ['D0103']  # start with one subject for speed; expand later
ACC_TRIALS_ONLY = False
N_JOBS = 4

CONDITION_LABEL = 'stimulus_experiment_conditions'

EPOCHS_ROOT_FILE = "Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit"

ROIS_DICT = {
    'lpfc': ["G_front_inf-Opercular", "G_front_inf-Orbital", "G_front_inf-Triangul",
             "G_front_middle", "G_front_sup", "Lat_Fis-ant-Horizont",
             "Lat_Fis-ant-Vertical", "S_circular_insula_ant", "S_circular_insula_sup",
             "S_front_inf", "S_front_middle", "S_front_sup"],
}

ELECTRODES = 'sig'
SAMPLING_RATE = 256
WINDOW_SIZE = 64
STEP_SIZE = 16
N_PERM = 100  # small for sandbox testing; bump to 1000 for the real run

In [10]:
if LAB_root is None:
    HOME = os.path.expanduser("~")
    USER = os.path.basename(HOME)
    if os.name == 'nt':
        LAB_root = os.path.join(HOME, "Box", "CoganLab")
    elif sys.platform == 'darwin':
        LAB_root = os.path.join(HOME, "Library", "CloudStorage", "Box-Box", "CoganLab")
    else:
        LAB_root = f"/cwork/{USER}" if os.path.exists(f"/cwork/{USER}") else os.path.join(HOME, "CoganLab")
print("LAB_root:", LAB_root)

LAB_root: /cwork/jz421


In [11]:
conditions = get_conditions_obj(CONDITION_LABEL)
condition_names = list(conditions.keys())
condition_comparisons = get_comparisons(CONDITION_LABEL)
subtraction_pairs = get_subtraction_pairs(CONDITION_LABEL)
anova_factors = get_anova_factors(CONDITION_LABEL)
anova_interactions = get_anova_interactions(CONDITION_LABEL)

print("Number of conditions:", len(conditions))
print("Condition names (first 4):", condition_names[:4])
print("ANOVA factors:", anova_factors)
print("Comparisons:", condition_comparisons)
print("Subtraction pairs:", subtraction_pairs)
print("Interactions:")
for i in anova_interactions:
    print("  ", i['name'], "->", i['factors'])

Number of conditions: 16
Condition names (first 4): ['Stimulus_i25s25', 'Stimulus_i25s75', 'Stimulus_i75s25', 'Stimulus_i75s75']
ANOVA factors: ['congruency', 'congruencyProportion', 'switchType', 'switchProportion']
Comparisons: {}
Subtraction pairs: []
Interactions:
   congruency_x_congruencyProportion -> ['congruency', 'congruencyProportion']
   switchType_x_switchProportion -> ['switchType', 'switchProportion']
   congruency_x_switchProportion -> ['congruency', 'switchProportion']
   switchType_x_congruencyProportion -> ['switchType', 'congruencyProportion']


In [12]:
config_dir = os.path.join(project_root, 'src', 'analysis', 'config')
subjects_electrodestoROIs_dict = utils.make_or_load_subjects_electrodes_to_ROIs_dict(
    subjects=SUBJECTS, save_dir=config_dir,
    filename='subjects_electrodestoROIs_dict.json',
)

sig_chans_per_subject = utils.get_sig_chans_per_subject(
    SUBJECTS, EPOCHS_ROOT_FILE, task=TASK, LAB_root=LAB_root,
)

rois = list(ROIS_DICT.keys())
all_electrodes_per_subject_roi, sig_electrodes_per_subject_roi = \
    utils.make_sig_electrodes_per_subject_and_roi_dict(
        ROIS_DICT, subjects_electrodestoROIs_dict, sig_chans_per_subject,
    )

print("rois:", rois)
print("Per-ROI electrode counts (sig):")
for r in rois:
    for s in SUBJECTS:
        print(f"  {r} / {s}: {len(sig_electrodes_per_subject_roi[r].get(s, []))} sig elecs")

Attempting to load the subjects' electrodes-to-ROIs dictionary...
Loaded data from /hpc/group/coganlab/jz421/GlobalLocal/src/analysis/config/subjects_electrodestoROIs_dict.json
Dictionary loaded successfully. Ready to proceed!
Loaded significant channels for subject D0103
For subject D0057, G_front_inf-Opercular, G_front_inf-Orbital, G_front_inf-Triangul, G_front_middle, G_front_sup, Lat_Fis-ant-Horizont, Lat_Fis-ant-Vertical, S_circular_insula_ant, S_circular_insula_sup, S_front_inf, S_front_middle, S_front_sup electrodes are: ['RAI6', 'RAI12', 'RAI13', 'RAI14', 'RAI15', 'RAI16', 'RPI15', 'RPI14', 'RAMF10', 'RAMF11', 'RAMF12', 'RAMF13', 'RAMF14', 'RAIF11', 'RAIF12', 'RAIF13', 'RAIF14']
For subject D0059, G_front_inf-Opercular, G_front_inf-Orbital, G_front_inf-Triangul, G_front_middle, G_front_sup, Lat_Fis-ant-Horizont, Lat_Fis-ant-Vertical, S_circular_insula_ant, S_circular_insula_sup, S_front_inf, S_front_middle, S_front_sup electrodes are: ['LMMF9', 'LMMF11', 'LMMF10', 'LMMF12', 'LP

In [13]:
subjects_mne_objects = utils.create_subjects_mne_objects_dict(
    subjects=SUBJECTS, epochs_root_file=EPOCHS_ROOT_FILE,
    conditions=conditions, task=TASK,
    just_HG_ev1_rescaled=True, acc_trials_only=ACC_TRIALS_ONLY,
)

# Inspect: what's in there?
sub0 = SUBJECTS[0]
cond0 = condition_names[0]
print("Top-level keys per subject:", list(subjects_mne_objects[sub0].keys())[:5], "...")
print(f"Per-condition keys ({sub0}/{cond0}):",
      list(subjects_mne_objects[sub0][cond0].keys()))
epochs0 = subjects_mne_objects[sub0][cond0]['HG_ev1_power_rescaled']
print("Epochs shape:", epochs0.get_data().shape, "(n_trials, n_channels, n_times)")
print("Times:", epochs0.times[0], "→", epochs0.times[-1], f"({len(epochs0.times)} samples)")

Loading data for subject: D0103
Reading /cwork/jz421/BIDS-1.1_GlobalLocal/BIDS/derivatives/freqFilt/figs/D0103/D0103_Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit_HG_ev1_rescaled-epo.fif ...
    Found the data of interest:
        t =   -1000.00 ...    1500.00 ms
        0 CTF compensation matrices available
Adding metadata with 29 columns
448 matching events found
No baseline correction applied
0 projection items activated
Replacing existing metadata with 29 columns
Reading /cwork/jz421/BIDS-1.1_GlobalLocal/BIDS/derivatives/freqFilt/figs/D0103/D0103_Stimulus_-1.0to1.5sec_0.5sec_within-1.0-0.0sec_base_decFactor_8_outliers_10_drop_thresh_perc_5.0_70.0-150.0_Hz_padLength_0.5s_stat_func_ttest_ind_equal_var_False_nan_policy_omit_HG_ev1_power_rescaled-epo.fif ...
    Found the data of interest:
        t =   -1000.00 ...    1500.00 ms
        0 CTF compensation

/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)
/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


Adding metadata with 30 columns
20 matching events found
No baseline correction applied
    Stimulus_i25s75: 20 valid trials out of 20
  Loading condition: Stimulus_i75s25
Adding metadata with 30 columns
20 matching events found
No baseline correction applied
    Stimulus_i75s25: 20 valid trials out of 20


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


  Loading condition: Stimulus_i75s75
Adding metadata with 30 columns
64 matching events found
No baseline correction applied


...

/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


    Stimulus_c25r75: 21 valid trials out of 21
  Loading condition: Stimulus_c75r25
Adding metadata with 30 columns
21 matching events found
No baseline correction applied


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


    Stimulus_c75r25: 21 valid trials out of 21
  Loading condition: Stimulus_c75r75
Adding metadata with 30 columns
9 matching events found
No baseline correction applied
    Stimulus_c75r75: 9 valid trials out of 9
Top-level keys per subject: ['Stimulus_i25s25', 'Stimulus_i25s75', 'Stimulus_i75s25', 'Stimulus_i75s75', 'Stimulus_i25r25'] ...
Per-condition keys (D0103/Stimulus_i25s25): ['HG_ev1_rescaled', 'HG_ev1_rescaled_avg', 'HG_ev1_rescaled_std_err', 'HG_ev1_power_rescaled', 'HG_ev1_power_rescaled_avg', 'HG_ev1_power_rescaled_std_err']
Epochs shape: (3, 222, 641) (n_trials, n_channels, n_times)
Times: -1.0 → 1.5 (641 samples)


/hpc/group/coganlab/jz421/GlobalLocal/src/analysis/utils/general_utils.py:511: RuntimeWarning: Concatenation of Annotations within Epochs is not supported yet. All annotations will be dropped.
  event_epochs = mne.concatenate_epochs(combined_epochs_list)


In [14]:
if ELECTRODES == 'all':
    raw_electrodes = all_electrodes_per_subject_roi
    elec_string_to_add_to_filename = 'all_elecs'
else:
    raw_electrodes = sig_electrodes_per_subject_roi
    elec_string_to_add_to_filename = 'sig_elecs'

electrodes = utils.filter_electrode_lists_against_subjects_mne_objects(
    rois, raw_electrodes, subjects_mne_objects,
)

dropped, _ = utils.find_difference_between_two_electrode_lists(raw_electrodes, electrodes)
total_dropped = sum(len(v) for d in dropped.values() for v in d.values())
print(f"Total dropped electrodes: {total_dropped}")
print("Final electrodes per ROI per subject:")
for r in rois:
    for s in SUBJECTS:
        print(f"  {r} / {s}: {len(electrodes[r].get(s, []))} electrodes")

Total dropped electrodes: 0
Final electrodes per ROI per subject:
  lpfc / D0103: 9 electrodes


In [15]:
evks_dict_elecs = make_multi_channel_evokeds_for_all_conditions_and_rois(
    subjects_mne_objects, SUBJECTS, rois, condition_names, electrodes,
)

# Inspect
roi0 = rois[0]
cond0 = condition_names[0]
evk = evks_dict_elecs[cond0][roi0]
print(f"Evoked for {cond0} / {roi0}:")
print(f"  data shape: {evk.data.shape}  (n_electrodes, n_times)")
print(f"  ch_names (first 3): {evk.ch_names[:3]}")
print(f"  times: {evk.times[0]:.3f} → {evk.times[-1]:.3f}")

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy functi

In [21]:
evks_dict_elecs

{'Stimulus_i25s25': {'lpfc': <Evoked | '0.33 × Stimulus/i25.0/s25.0/BigLetterh/SmallLetters/Taskg/TargetLetterh/Responded1.0/ParticipantResponse115.0/CorrectResponse115.0/TrialCount369.0/BlockTrialCount33.0/ReactionTime866.0/Accuracy1.0/D103 + 0.33 × Stimulus/i25.0/s25.0/BigLetters/SmallLetterh/Taskg/TargetLetters/Responded1.0/ParticipantResponse114.0/CorrectResponse114.0/TrialCount391.0/BlockTrialCount55.0/ReactionTime716.0/Accuracy1.0/D103 + 0.33 × Stimulus/i25.0/s25.0/BigLetters/SmallLetterh/Taskl/TargetLetterh/Responded1.0/ParticipantResponse115.0/CorrectResponse115.0/TrialCount343.0/BlockTrialCount7.0/ReactionTime1416.0/Accuracy1.0/D103' (average, N=3), -1 – 1.5 s, baseline off, 9 ch, ~57 KiB>},
 'Stimulus_i25s75': {'lpfc': <Evoked | '0.05 × Stimulus/i25.0/s75.0/BigLetterh/SmallLetters/Taskg/TargetLetterh/Responded1.0/ParticipantResponse114.0/CorrectResponse115.0/TrialCount243.0/BlockTrialCount19.0/ReactionTime1133.0/Accuracy0.0/D103 + 0.05 × Stimulus/i25.0/s75.0/BigLetterh/SmallL

In [23]:
windowed_data = process_windowed_data_for_anova(
    subjects_mne_objects, condition_names, rois, SUBJECTS,
    electrodes, window_size=WINDOW_SIZE,
    step_size=STEP_SIZE, sampling_rate=SAMPLING_RATE,
)

# Inspect
arr0 = windowed_data[cond0][roi0][0]  # subject 0
print(f"windowed_data[{cond0}][{roi0}][sub0] shape:")
print(f"  {arr0.shape}  (n_trials, n_windows, n_channels)")
n_windows_expected = (len(evk.times) - WINDOW_SIZE) // STEP_SIZE + 1
print(f"  expected n_windows: {n_windows_expected}")

NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy functi

In [32]:
ref_evk = next(
    (evks_dict_elecs[c][r] for c in condition_names for r in rois
     if evks_dict_elecs[c][r] is not None and evks_dict_elecs[c][r].data.shape[0] > 0),
    None,
)
full_times = ref_evk.times

df = create_windowed_anova_dataframe(
    windowed_data, conditions, rois, electrodes_per_subject_roi=electrodes, subjects=SUBJECTS,
    times=full_times, window_size=WINDOW_SIZE,
    step_size=STEP_SIZE, sampling_rate=SAMPLING_RATE,
)

print("DataFrame shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nHead:")
df.head()

DataFrame shape: (147852, 12)
Columns: ['SubjectID', 'Electrode', 'ROI', 'WindowCenter', 'WindowIndex', 'Trial', 'Activity', 'BIDS_events', 'congruency', 'congruencyProportion', 'switchType', 'switchProportion']

Head:


,SubjectID,Electrode,ROI,WindowCenter,WindowIndex,Trial,Activity,BIDS_events,congruency,congruencyProportion,switchType,switchProportion
0,D0103,LFAM8,lpfc,-0.875,0,1,-0.116866,[Stimulus/i25.0/s25.0],i,75%,s,25%
1,D0103,LFAM9,lpfc,-0.875,0,1,0.212812,[Stimulus/i25.0/s25.0],i,75%,s,25%
2,D0103,LAI13,lpfc,-0.875,0,1,-0.716582,[Stimulus/i25.0/s25.0],i,75%,s,25%
3,D0103,LAI14,lpfc,-0.875,0,1,-0.077901,[Stimulus/i25.0/s25.0],i,75%,s,25%
4,D0103,LAI4,lpfc,-0.875,0,1,0.353719,[Stimulus/i25.0/s25.0],i,75%,s,25%


In [ ]:
def main(args):





    # ------------------------------------------------------------------
    # 4. Build evokeds for all conditions
    # ------------------------------------------------------------------
    evks_dict_elecs = make_multi_channel_evokeds_for_all_conditions_and_rois(
        subjects_mne_objects, args.subjects, rois, condition_names, electrodes
    )

    # ------------------------------------------------------------------
    # 5. Statistical testing
    # ------------------------------------------------------------------
    significant_clusters = {}
    interaction_results = None

    if args.statistical_method == 'time_perm_cluster':
        print("\nRunning statistical tests comparing TWO conditions")
        if len(condition_names) != 2:
            raise ValueError("Time perm cluster stats requires exactly two conditions.")

        p_values_dict = {}
        condition1_name, condition2_name = condition_names

        for roi in rois:
            print(f"-- Processing ROI: {roi} --")
            try:
                evoked_cond1_this_roi = evks_dict_elecs[condition1_name][roi]
                evoked_cond2_this_roi = evks_dict_elecs[condition2_name][roi]

                mask_roi, p_values = time_perm_cluster_between_two_evokeds(
                    evoked_cond1_this_roi, evoked_cond2_this_roi,
                    p_thresh=args.p_thresh_for_time_perm_cluster_stats,
                    p_cluster=args.p_cluster, n_perm=args.n_perm,
                    tails=args.tails, axis=0, stat_func=args.stat_func,
                    ignore_adjacency=None,
                    permutation_type=args.permutation_type,
                    vectorized=True, n_jobs=args.n_jobs, seed=None, verbose=True,
                )

                significant_clusters[roi] = mask_roi
                p_values_dict[roi] = p_values
            except KeyError:
                print(f"   Skipping ROI {roi}: Missing prepared evoked data for "
                      f"one or both conditions.")

    elif args.statistical_method == 'anova':
        if not anova_interactions:
            raise ValueError(
                f"condition_label '{condition_label}' has no 'anova_interactions' "
                f"in the registry. Add them to enable ANOVA cluster correction."
            )
        print(f"\nRunning ANOVA-based 2-way interaction cluster correction for "
              f"{len(anova_interactions)} interaction(s) across {len(rois)} ROI(s)")
        interaction_results = run_anova_interaction_clusters(
            evks_dict_elecs, conditions, anova_interactions, rois,
            p_thresh=args.p_thresh_for_time_perm_cluster_stats,
            cluster_forming_p=args.p_cluster,
            n_perm=args.n_perm, tails=args.tails,
            seed=None, verbose=True,
        )

    # ------------------------------------------------------------------
    # 6. Plotting
    # ------------------------------------------------------------------
    plot_power_traces_for_all_rois(
        evks_dict_elecs, rois, condition_names, conditions_save_name, plot_params,
        args.window_size, args.sampling_rate,
        significant_clusters=significant_clusters or None,
        save_dir=save_dir,
        error_type='sem',
        plot_style=args.plot_style,
        save_name_suffix=elec_string_to_add_to_filename,
    )

    # If we ran the ANOVA cluster correction, draw the 16-condition mega-plot
    # with 4 stacked interaction-cluster bars + one 4-trace plot per interaction.
    if interaction_results is not None:
        plot_anova_interaction_results(
            evks_dict_elecs, conditions, condition_names, conditions_save_name,
            rois, anova_interactions, interaction_results,
            plot_style=args.plot_style, save_dir=save_dir,
            save_name_suffix=elec_string_to_add_to_filename, error_type='sem',
        )

    # ------------------------------------------------------------------
    # 7. Subtraction-pair plotting (driven by registry)
    # ------------------------------------------------------------------
    if subtraction_pairs:
        # All names needed must be present in evks_dict_elecs
        usable_pairs = [p for p in subtraction_pairs
                        if p[0] in evks_dict_elecs and p[1] in evks_dict_elecs]
        if usable_pairs:
            subtracted_evks_dict_elecs = create_subtracted_evokeds_dict(
                evks_dict_elecs, usable_pairs, rois
            )
            sub_pair_names = ['-'.join(pair) for pair in usable_pairs]
            sub_save_name = (f"{condition_label}_subtractions_"
                             f"{args.epochs_root_file}_"
                             f"{len(args.subjects)}_subjects")
            plot_power_traces_for_all_rois(
                subtracted_evks_dict_elecs, rois, sub_pair_names, sub_save_name,
                plot_params, save_dir=save_dir,
                window_size=args.window_size, sampling_rate=args.sampling_rate,
                error_type='sem',
                plot_style=args.plot_style,
                save_name_suffix=elec_string_to_add_to_filename,
            )
        else:
            print("[subtraction] No usable subtraction pairs (condition keys "
                  "missing from evks_dict).")

    # ------------------------------------------------------------------
    # 8. Save results
    # ------------------------------------------------------------------
    results_save_dir = os.path.join(save_dir, 'saved_results')
    os.makedirs(results_save_dir, exist_ok=True)

    for condition_name in condition_names:
        for roi in rois:
            evk = evks_dict_elecs[condition_name][roi]
            if evk is not None:
                np.savez(
                    os.path.join(
                        results_save_dir,
                        f'{conditions_save_name}_{condition_name}_{roi}_evoked.npz'
                    ),
                    data=evk.data, times=evk.times, ch_names=evk.ch_names
                )

    if significant_clusters:
        np.savez(
            os.path.join(results_save_dir,
                         f'{conditions_save_name}_significant_clusters.npz'),
            **significant_clusters
        )

    if interaction_results is not None:
        # Persist as one file per ROI / interaction (masks + p values)
        for roi, by_inter in interaction_results.items():
            for inter_name, info in by_inter.items():
                np.savez(
                    os.path.join(
                        results_save_dir,
                        f'{conditions_save_name}_{roi}_{inter_name}_interaction_cluster.npz'
                    ),
                    mask=info['mask'],
                    t_obs=info['t_obs'],
                    cluster_p_values=info['cluster_p_values'],
                )

    metadata = {
        'condition_label': condition_label,
        'condition_names': condition_names,
        'rois': rois,
        'conditions_save_name': conditions_save_name,
        'sfreq': args.sampling_rate,
        'statistical_method': args.statistical_method,
        'anova_factors': anova_factors,
        'anova_interactions': [
            {'name': i['name'], 'factors': i['factors'],
             'label': i.get('label', i['name'])}
            for i in anova_interactions
        ] if anova_interactions else [],
    }
    with open(os.path.join(results_save_dir,
                           f'{conditions_save_name}_metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=2)


if __name__ == "__main__":
    if len(sys.argv) == 1:
        pass
    else:
        print("This script should be called via run_power_traces_dcc.py")
        print("Direct command-line execution is not supported with complex parameters.")
        sys.exit(1)